# CRASH LENS - Virginia Data Sync

This notebook downloads crash data from Virginia Roads and uploads to Google Drive.

**Steps:**
1. Test if Virginia Roads API is accessible from Colab
2. Download crash data
3. Save to Google Drive
4. (Your app loads from Google Drive)

---

## Step 0: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_FOLDER = '/content/drive/MyDrive/CRASH_LENS_Data'
os.makedirs(DRIVE_FOLDER, exist_ok=True)
print(f"Drive mounted! Data will be saved to: {DRIVE_FOLDER}")

## Step 1: Test API Access
Check if Virginia Roads API is accessible from Google Colab.

In [ ]:
import requests

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
    'Referer': 'https://virginiaroads-vdot.opendata.arcgis.com/',
}

def test_api_access():
    # Check our IP
    try:
        ip_info = requests.get('https://ipinfo.io/json', timeout=10).json()
        print(f"Colab IP: {ip_info.get('ip')}")
        print(f"Organization: {ip_info.get('org')}")
        print()
    except:
        pass
    
    # Test ArcGIS API
    print("Testing ArcGIS API...")
    api_url = "https://services.arcgis.com/p5v98VHDX9Atv3l7/arcgis/rest/services/CrashData_Basic_Updated/FeatureServer/0/query"
    params = {'where': '1=1', 'returnCountOnly': 'true', 'f': 'json'}
    
    try:
        response = requests.get(api_url, params=params, headers=HEADERS, timeout=30)
        if response.status_code == 200:
            data = response.json()
            if 'count' in data:
                print(f"SUCCESS! Records available: {data['count']:,}")
                return True
        print(f"BLOCKED - Status: {response.status_code}")
        return False
    except Exception as e:
        print(f"Error: {e}")
        return False

def test_csv_access():
    print("\nTesting CSV Download...")
    csv_url = "https://www.virginiaroads.org/api/download/v1/items/1a96a2f31b4f4d77991471b6cabb38ba/csv?layers=0"
    try:
        response = requests.head(csv_url, headers=HEADERS, timeout=30)
        if response.status_code == 200:
            print("SUCCESS! CSV is accessible")
            return True
        print(f"BLOCKED - Status: {response.status_code}")
        return False
    except Exception as e:
        print(f"Error: {e}")
        return False

print("="*50)
print("  VIRGINIA ROADS API ACCESS TEST")
print("="*50)
api_ok = test_api_access()
csv_ok = test_csv_access()
print("\n" + "="*50)
if api_ok or csv_ok:
    print("  SUCCESS: API is accessible from Colab!")
else:
    print("  BLOCKED: Use local PC download instead.")
print("="*50)

## Step 2: Download Crash Data
If test passed, run this to download data.

In [ ]:
import pandas as pd
import io
from datetime import datetime

# Configuration - Change these for your jurisdiction
JURISDICTION = 'henrico'
JURISDICTION_CODE = '44'

def download_crash_data():
    csv_url = "https://www.virginiaroads.org/api/download/v1/items/1a96a2f31b4f4d77991471b6cabb38ba/csv?layers=0"
    print("Downloading crash data (this may take a few minutes)...")
    response = requests.get(csv_url, headers=HEADERS, timeout=300)
    response.raise_for_status()
    df = pd.read_csv(io.StringIO(response.text))
    print(f"Downloaded {len(df):,} total records")
    return df

def filter_jurisdiction(df, juris_code):
    for col in ['Juris_Code', 'JURIS_CODE', 'juris_code']:
        if col in df.columns:
            mask = df[col].astype(str).str.strip() == str(juris_code)
            filtered = df[mask].copy()
            print(f"Filtered to {len(filtered):,} {JURISDICTION} records")
            return filtered
    return df

try:
    df_all = download_crash_data()
    df_filtered = filter_jurisdiction(df_all, JURISDICTION_CODE)
    print("\nPreview:")
    display(df_filtered.head())
except Exception as e:
    print(f"Download failed: {e}")
    print("Virginia Roads may be blocking Colab.")

## Step 3: Save to Google Drive

In [ ]:
timestamp = datetime.now().strftime('%Y-%m-%d')

output_file = f"{DRIVE_FOLDER}/crashes.csv"
df_filtered.to_csv(output_file, index=False)
print(f"Saved: {output_file}")

backup_file = f"{DRIVE_FOLDER}/crashes_{timestamp}.csv"
df_filtered.to_csv(backup_file, index=False)
print(f"Backup: {backup_file}")

size_mb = os.path.getsize(output_file) / (1024 * 1024)
print(f"\nFile size: {size_mb:.1f} MB")
print(f"Records: {len(df_filtered):,}")

## Step 4: Get Public Link for Your App

1. Go to Google Drive
2. Find `CRASH_LENS_Data/crashes.csv`
3. Right-click → Share → Anyone with link → Viewer
4. Copy link: `https://drive.google.com/file/d/XXXXX/view`
5. Convert to direct download: `https://drive.google.com/uc?export=download&id=XXXXX`

Use that URL in your app to fetch the data!